# Python程序设计（项目实战） · 工单P4.1《多源数据采集系统》
**第6周 | 2课时(80分钟) | 理实一体**  | 教师参考版

| 层 | 任务定位 | 达标要求 |
|----|---------|---------|
| **A 师傅** | 进阶拓展 | 采集率统计面板：实时打印成功率/延迟/丢弃数（简易APM）；示范讲解 |
| **B 组长** | 标准达成 | 断线重连实测通过 |
| **C 操作员** | 基础完成 | 三源贯通+采集率>95% |
| **D 学徒** | 跟做保底 | 跟做示例跑通任务1，程序出结果 |
> 每4周按工单成绩动态升降层（连续2次≥B升一层）

> **本教师版已补全 3 个任务空位（对应学生版 read_one / on_message / DataProcessor.run 的 3 处占位）**，
> 并在**真协议库缺失或 PLC/MQTT 端口探活失败**时自动降级为**进程内虚拟产线**（同 API、同报文格式），
> 三线程照常并发跑：退避重连、JSON 解析、幂等去重、振动告警、SQLite 批量入库、对账恒等式全部可现场验证。


## 一、课前资讯（Predict·预测）
> **预习快问**：为什么‘共享list.append’有隐患而Queue安全？

> **课前检查**：模板仓库已克隆更新；导学单与实施计划已提交平台

## 二、PRIMM五步法（先读后写）
本工单按国际编程教学法 **PRIMM** 组织：
1. **Predict 预测**——先运行下方示例，猜输出再验证；
2. **Run 运行**——跑通环境与数据；
3. **Investigate 研究**——读懂starter每段在做什么；
4. **Modify 修改**——把 starter 的 3 处待填空位补全（本教师版已全部给出完成态）；
5. **Make 创造**——独立完成工单任务。


In [3]:
# --- 环境与数据准备(勿改, 运行即可) ---
import os, sys, subprocess
os.chdir(os.path.abspath("../"))          # 切到课程包根目录(data/所在处)
print("工作目录:", os.getcwd())


工作目录: /Users/joelgong/PycharmProjects/ai-course-teaching-resources/03-Python程序设计-项目实战


## 三、任务要求（Make）
**实施步骤**
1. 启动plc_simulator三台PLC
2. ModbusCollector轮询0.5s×3台(÷10还原)
3. read_one退避重试(1→2→4s)

**验收标准**（工单《P4》「验收标准」四行原文）：`及格(D)`＝`三源数据贯通；采集率>95%；无数据丢失；系统文档` ｜ `良好(C)`＝`断线重连实测通过；统计输出准确` ｜ `优秀(B)`＝`采集率≥99.5%；延迟<100ms；线程安全(无共享可变状态)` ｜ `超优(A)`＝`优秀+故障自动转移/主从切换或性能监控小面板`
> **口径说明（别把两套字母混为一谈）**：上四行是**工单《P4》「验收标准」原文**（及格/良好/优秀/超优），与本课课堂分层（D 跟做保底 / C 基础完成 / B 标准达成 / A 进阶拓展）**不同轴**——课堂 C≈工单及格、课堂 B≈工单良好，工单「优秀」档（如 采集率≥99.5%；延迟<100ms；线程安全(无共享可变状态)）课堂阶梯不覆盖。工程化项目的 `output/report.txt` 已按工单原文逐行取证：p11/p21/p33/p55 直接以这四行为分层口径，p12/p22/p41/p66 另设「工单验收四档对照」节；本机取不了证的子项逐条点名，不假判。


### Starter 1：`导入块 + 参数 + 协议库探活选择（真库 / 进程内虚拟产线）`
（教师版：下方为完成态可运行代码；学生版此处为待填空位）
> 关键工程判断：**装了库 ≠ 连得上设备**。cell 5 先 TCP 探活 5020~5022 / 1883，再决定用 `pymodbus`/`paho` 还是同名替身，并把选择**打印出来**——这就是工单要求的「来源可追」。


In [6]:
# -*- coding: utf-8 -*-
"""P4.1 多源工业数据采集系统 —— 三线程架构骨架(已给, 聚焦协议集成与可靠性)
前置: python ../工具/plc_simulator.py  (三台PLC: 5020/5021/5022)
     python ../02-程序设计基础/虚拟产线/mini_mqtt_broker.py
教师版完成态: 真协议库缺失 **或产线端口探活失败** 时, 自动降级为"进程内虚拟产线"(同名同API替身),
             三线程照常真实并发 —— 寄存器值/JSON状态报文/退避重连/去重告警入库一样不少。
"""
import datetime
import json
import logging
import math
import queue
import random
import socket
import sqlite3
import sys
import threading
import time

# ---- 采集参数(与工单/点表一致, 讲解时先对齐口径再写代码) ----
PLCS = [("127.0.0.1", 5020), ("127.0.0.1", 5021), ("127.0.0.1", 5022)]
NAMES = ["temp_c", "press_mpa", "vib_mm_s", "rpm", "status"]
SCALE = 10                 # 寄存器 ×10 定点存储 → 统一 ÷10 还原工程值(status 是状态字不除)
RETRY_TIMES = 3            # 断线重连最多试 3 次
BACKOFF_CAP = 4            # 退避封顶 4 秒 → 1s→2s→4s
BACKOFF_UNIT = 1.0         # 退避时间单位: 课堂/生产 1.0 秒(下方自检会临时缩放到 0.05 以省时间)
DB_PATH = "buffer.db"      # SQLite 缓冲库(课程包根目录, P5 数仓的前置)
FLUSH_SECONDS = 5.0        # 批量入库兜底间隔(工单: 每50条 或 每5秒)
GET_TIMEOUT = 1.0          # q.get 心跳超时(骨架原意5秒): 缩到1秒 → stop 后线程秒退, 最后一批buf及时落盘
RUN_SECONDS = 5.0          # notebook 演示时长; 课堂按骨架原意为 30 秒(见 Starter 5 的说明)
PUB_INTERVAL = 0.8         # 离线 MQTT 模拟的上报间隔(秒)
FLAKY_PORT, OUTAGE_EVERY = 5021, 3     # 2号PLC随机掉线模拟: 每3次连接拒1次
VIB_LIMIT, TEMP_LIMIT = 4.5, 90.0      # 告警阈值: 振动mm/s / 温度℃


class _LiveStderr:
    """notebook 逐 cell 重定向 stderr 时, 让日志句柄动态写到"当前" stderr
    (否则日志会在 cell 5 就被绑死, 后续 cell 的告警看不到)"""

    def write(self, s): sys.stderr.write(s)
    def flush(self): sys.stderr.flush()
    def isatty(self): return False
    def close(self): pass


logging.basicConfig(level=logging.INFO, stream=_LiveStderr(),
                    format="%(asctime)s.%(msecs)03d %(threadName)-7s %(levelname)s %(message)s",
                    datefmt="%H:%M:%S")
log = logging.getLogger("collector")

try:                       # 课堂主路径: 真协议库(需先起 工具/plc_simulator.py)
    from pymodbus.client import ModbusTcpClient as RealModbusClient
    HAVE_MODBUS = True
except ImportError:        # 缺库: 下面给出进程内 Modbus 替身(寄存器口径同 mini_modbus_server)
    RealModbusClient = None
    HAVE_MODBUS = False

class _SimRegisters:
    """模仿 pymodbus 响应对象: .registers(list) / .isError()"""

    def __init__(self, regs):
        self.registers, self._err = regs, False

    def isError(self):
        return self._err

class SimModbusClient:
    """40001温度℃ 40002压力MPa 40003振动mm/s 40004转速 40005状态字 —— 均 ×10 定点
    可靠性演示: 2号PLC(5021)每 OUTAGE_EVERY 次连接拒 1 次 → 正好触发 read_one 的退避重连"""
    _calls, _t0 = {}, time.time()

    def __init__(self, host="127.0.0.1", port=5020, **kw):
        self.host, self.port = host, port

    def connect(self):
        n = self._calls.get(self.port, 0) + 1
        self._calls[self.port] = n
        if self.port == FLAKY_PORT and n % OUTAGE_EVERY == 0:
            return False                     # 等价于真实环境的 ConnectionRefused/超时
        return True

    def read_holding_registers(self, address=0, count=5, **kw):
        t = time.time() - self._t0
        u = {5020: 1, 5021: 2, 5022: 3}.get(self.port, 1)
        temp = 55 + 15 * math.sin(t / 20 + u)                # ℃
        press = 0.5 + 0.2 * math.sin(t / 15)                 # MPa(点表口径 0.1~0.9)
        vib = 42 + 12 * math.sin(t * 1.1 + u * 2.4)          # ×10 → 3.0~5.4 mm/s(秒级起伏, 便于演示告警)
        rpm = 1480 + 40 * math.sin(t / 30)                   # 真实PLC按 rpm/10 存
        regs = [int(temp * SCALE), int(press * SCALE), int(vib),
                int(rpm / SCALE), 1 if vib / SCALE > VIB_LIMIT else 0]
        return _SimRegisters(regs[address:address + count])

    def close(self):
        return None

# Client 用真库还是替身, 由下方「探活」块决定

try:                       # 课堂主路径: 真 paho-mqtt(需先起 mini_mqtt_broker.py)
    import paho.mqtt.client as real_mqtt
    HAVE_MQTT = True
except ImportError:        # 缺库: 下面给出进程内 broker+发布者(主题/报文与 02包 w11_collector 同构)
    real_mqtt = None
    HAVE_MQTT = False

class _SimMsg:
    """模仿 paho 的 msg 对象: .topic(str) / .payload(bytes)"""

    def __init__(self, topic, payload):
        self.topic, self.payload = topic, payload

class SimMqttClient:
    """最小 paho 替身: connect/subscribe/loop_forever + on_message(_c,_u,msg) 三参签名
    刻意每 2 轮重复投递同一报文(QoS0 语义的双发) → 正好演示下游幂等去重"""
    DEVICES = ["DEV-A1", "DEV-A2", "DEV-A3"]

    def __init__(self, *a, **kw):
        self.on_message = None
        self._topics, self._stop = [], threading.Event()
        self.published = 0

    def connect(self, host="127.0.0.1", port=1883, timeout=5):
        print(f"[离线MQTT] 已连接进程内代理 {host}:{port}(真实环境先起 mini_mqtt_broker.py)")
        return 0

    def subscribe(self, topic, qos=0, **kw):
        self._topics.append(topic)
        return (0, qos)

    def stop(self):
        self._stop.set()

    def loop_forever(self, *a, **kw):
        deadline = time.time() + RUN_SECONDS        # 真 paho 长驻; 模拟源按时自停, 便于反复运行
        rnd = random.Random(2026)
        n = 0
        while not self._stop.is_set() and time.time() < deadline:
            n += 1
            for i, dev in enumerate(self.DEVICES, start=1):
                now = time.time()
                temp = round(60 + 20 * math.sin(now / 18 + i) + rnd.uniform(-2, 2), 1)
                vib = round(4.2 + 1.0 * math.sin(now * 0.5 + i) + rnd.uniform(-0.3, 0.3), 2)
                metrics = {"temp_c": temp, "press_mpa": round(0.5 + 0.1 * i, 2),
                           "vib_mm_s": vib, "rpm": 1480 + 20 * i,
                           "status": 1 if vib > VIB_LIMIT else 0}
                payload = json.dumps({"device_id": dev, "ts": round(now, 3),
                                      "metrics": metrics}, ensure_ascii=False)
                for f in self._topics:              # device/+/status 通配 → 三台都命中
                    if self.on_message:
                        self.on_message(self, None, _SimMsg(f, payload.encode("utf-8")))
                        self.published += 1
                        if n % 2 == 0:              # 重复投递: 同 topic 同 payload(同 ts)
                            self.on_message(self, None, _SimMsg(f, payload.encode("utf-8")))
                            self.published += 1
            time.sleep(PUB_INTERVAL)

class _MqttModule:
    Client = SimMqttClient

    class CallbackAPIVersion:                        # paho 2.x 同名枚举(便于代码平移)
        VERSION1, VERSION2 = 1, 2

# ---- 数据源选择: 装了库 ≠ 产线在跑, 必须真探端口(pp03 装了 pymodbus/paho 但 PLC 没起) ----
def _port_alive(host, port, timeout=0.3):
    """TCP 探活: 连得上端口才走真协议; 连不上就用进程内虚拟产线(API 完全同名)"""
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except OSError:
        return False


MQTT_PORT = 1883
LIVE_PLC = [p for _h, p in PLCS if _port_alive("127.0.0.1", p)]
LIVE_MQTT = _port_alive("127.0.0.1", MQTT_PORT)
USE_MODBUS = HAVE_MODBUS and len(LIVE_PLC) == len(PLCS)     # 三台齐在线才算真链路
USE_MQTT = HAVE_MQTT and LIVE_MQTT

Client = RealModbusClient if USE_MODBUS else SimModbusClient
mqtt = real_mqtt if USE_MQTT else _MqttModule()


def _to_epoch(v):
    """时间戳归一化 → float 秒(下游按 (source, round(ts,1)) 去重, 必须是数值)
    02包上报用的是 'HH:MM:SS' 字符串 → 这里兼容三种写法, 缺字段则按采集时刻兜底"""
    if isinstance(v, (int, float)):
        return float(v)
    if isinstance(v, str):
        try:                                             # "HH:MM:SS" → 当天时刻
            return float(time.mktime(time.strptime(v, "%H:%M:%S")))
        except (ValueError, OverflowError):
            pass
        try:                                             # ISO 完整时间
            return datetime.datetime.fromisoformat(v).timestamp()
        except ValueError:
            pass
    return time.time()


print(f"协议库: pymodbus={'已安装' if HAVE_MODBUS else '未安装'} / "
      f"paho-mqtt={'已安装' if HAVE_MQTT else '未安装'}")
print(f"PLC 数据源: {'真协议 pymodbus ← 三台在线' if USE_MODBUS else '进程内虚拟PLC(5020/5021/5022)'} "
      f"{'(库已装但端口探活失败: ' + str(len(LIVE_PLC)) + '/3 在线)' if HAVE_MODBUS and not USE_MODBUS else ''}")
print(f"MQTT 数据源: {'真协议 paho-mqtt ← broker:1883 在线' if USE_MQTT else '进程内broker+三设备上报'} "
      f"{'(库已装但端口探活失败)' if HAVE_MQTT and not USE_MQTT else ''}")
print("想切真协议: 起 python 工具/plc_simulator.py 与 python ../02-程序设计基础/虚拟产线/mini_mqtt_broker.py, "
      "重跑本 cell 即可 —— 下方 read_one/on_message 一行都不用改")
print(f"演示时长 {RUN_SECONDS:.0f}s (课堂 30s)")


协议库: pymodbus=已安装 / paho-mqtt=已安装
PLC 数据源: 进程内虚拟PLC(5020/5021/5022) (库已装但端口探活失败: 0/3 在线)
MQTT 数据源: 进程内broker+三设备上报 (库已装但端口探活失败)
想切真协议: 起 python 工具/plc_simulator.py 与 python ../02-程序设计基础/虚拟产线/mini_mqtt_broker.py, 重跑本 cell 即可 —— 下方 read_one/on_message 一行都不用改
演示时长 5s (课堂 30s)


### Starter 2：`class ModbusCollector(threading.Thread)`
（教师版：下方为完成态可运行代码；学生版此处为待填空位）


In [8]:
class ModbusCollector(threading.Thread):
    """任务1(学生版第1空): 轮询三台PLC(0.5s), 读5寄存器÷10还原, 放入Queue
    可靠性: 连接失败退避重试(1s→2s→4s封顶); 2号PLC会随机掉线(实测点)"""

    def __init__(self, q: queue.Queue):
        super().__init__(daemon=True, name="Modbus")
        self.q, self.stop = q, threading.Event()
        # 教师版补充 retry/drop 计数: 支撑 B层"断线重连实测" 与 A层简易APM面板
        self.stat = {"ok": 0, "fail": 0, "retry": 0, "drop": 0}

    def read_one(self, host, port):
        # ===== 任务1 教师版完成态: Client(host=…, port=…) → read_holding_registers(address=0, count=5) =====
        # ⚠ pymodbus 3.x 必须用关键字参数(位置参数会 TypeError)
        # ⚠ 寄存器是 ×10 定点 → ÷SCALE 还原工程值; status 是状态字(0/1) 不除
        #    (40004 在真实PLC按 rpm/10 存, 统一÷10 得到的是 rpm÷100 —— 讲解时提醒学生核对点表)
        # ⚠ 断线重连: 最多 RETRY_TIMES 次, 间隔 2**i 秒 = 1s→2s→4s; 仍失败返回 None 交给 run() 全局退避
        for i in range(RETRY_TIMES):
            c = Client(host=host, port=port)
            try:
                if c.connect():                               # 连接失败(拒连/超时) 走下面的退避
                    rr = c.read_holding_registers(address=0, count=5)
                    if not rr.isError():                      # 异常响应(功能码+0x80) 也算本轮失败
                        out = {n: (v / SCALE if n != "status" else int(v))
                               for n, v in zip(NAMES, rr.registers)}
                        out["source"] = f"plc@{port}"          # 带端口=来源可追, 去重键的一半
                        out["ts"] = time.time()                # 数值秒时间戳, 与 MQTT 报文同构
                        if i:
                            log.info(f"PLC@{port} 第{i + 1}次尝试重连成功 ✓")
                        return out
                    log.warning(f"PLC@{port} 读到异常响应(isError=True)")
                else:
                    log.warning(f"PLC@{port} 第{i + 1}次连接被拒(掉线)")
            except Exception as e:                            # 报文错/socket错 → 不让线程猝死
                log.warning(f"PLC@{port} 第{i + 1}次读取异常: {type(e).__name__}: {e}")
            finally:
                c.close()                                     # 成败都要关socket, 否则文件描述符泄漏
            if i < RETRY_TIMES - 1:
                delay = min(2 ** i, BACKOFF_CAP) * BACKOFF_UNIT   # 1s → 2s → 4s(封顶)
                self.stat["retry"] += 1
                log.warning(f"PLC@{port} 退避 {delay:.2f}s 后重试")
                time.sleep(delay)
        return None

    def run(self):
        # ---- 骨架已给(勿改): 整轮全失败才把全局退避翻倍, 封顶4s; 正常节奏 0.5s 轮询 ----
        backoff = 1
        while not self.stop.is_set():
            got_any = False
            for host, port in PLCS:
                data = self.read_one(host, port)
                if data:
                    self.stat["ok"] += 1
                    try:
                        self.q.put(data, timeout=1)          # 带超时防永久阻塞(骨架已给)
                    except queue.Full:
                        self.stat["drop"] += 1               # 教师版补充: 背压丢弃计入对账
                    got_any = True
                else:
                    self.stat["fail"] += 1
            backoff = 1 if got_any else min(backoff * 2, 4)
            if not got_any:
                log.warning(f"三台PLC本轮全部无数据 → 全局退避 {backoff}s (1→2→4s封顶)")
            time.sleep(0.5 if got_any else backoff)
        log.info(f"Modbus线程收尾: 成功{self.stat['ok']} 失败{self.stat['fail']} "
                 f"退避重试{self.stat['retry']} 丢弃{self.stat['drop']}")


### 💡 分层提示（先自己试5分钟再看）
**思路提示**（一条对应学生版一个空位，按顺序做）
> - **任务1 `ModbusCollector.read_one`**：`c = Client(host=host, port=port)`（pymodbus 3.x **必须关键字参数**）→ `if c.connect():` → `rr = c.read_holding_registers(address=0, count=5)`；先 `rr.isError()` 判异常响应，再 `rr.registers`；工程值 = `寄存器值 / 10`（`status` 是状态字保留整数）；组装 `dict` 时补 `source=f"plc@{port}"` 和数值 `ts`；重试用 `for i in range(3)` + `time.sleep(min(2**i, 4))` 就是 **1s→2s→4s 封顶**；`finally: c.close()` 防句柄泄漏；全部失败 `return None`（不要抛异常，否则线程直接死掉）
> - **任务2 `MQTTSubscriber.on_message`**：`msg.payload` 是 **bytes** → `json.loads(msg.payload.decode("utf-8"))`；把 `metrics` 摊平成一层字典（下游取 `d["vib_mm_s"]` 才不用两套口径），再加 `source="mqtt"`；`ts` **必须归一成数值秒**，否则骨架给的 `round(d.get("ts",0),1)` 会 `TypeError`；入队用 `self.q.put(data, timeout=1)` 并 `try/except queue.Full` 计丢弃；整段包 `try/except` —— 一条脏报文不该弄死订阅线程
> - **任务3 `DataProcessor.run`**：`key=(source, round(ts,1))` 命中 `seen` 就 `continue` 并计数（**幂等去重**，重试/QoS0双发都靠它）；`vib > 4.5` → `log.warning` 告警 + 计数；行 `(source, ts, temp, vib)` 追加进 `self.buf`，`len(buf) >= batch` 或距上次超过 5 秒就 `flush(db)`（工单：每50条或每5秒）；线程退出前**必须再 flush 一次**并 `db.close()`，否则最后一批留在内存里=丢数
> - **去重键的坑**：骨架给的 `(source, round(ts,1))` 对 MQTT 这一路会误伤——`source` 全是 "mqtt"，三台设备同一 0.1 秒桶就把正常数据当重复丢了；幂等键要能**唯一标识一条数据**，补 `device_id` 才对（写进工单"参数-收益"留痕）
> - **可靠性加分项（B/A 层）**：给 `DataProcessor` 加 `stop` 事件做优雅收尾（工单知识卡 B05："要 flush 的线程不能只靠 daemon"）；把 `发送=接收+在途+丢弃` 写成断言（对账恒等式），差一条就报警

| 层 | 这样做 |
|----|--------|
| **D 学徒** | 跟教师演示逐步照做：定位完成态 cell 里的「任务n」标记行 → 按上面思路逐句翻译成代码 → 运行看输出 → 对不上就看下方"报错自查" |
| **C 操作员** | 只看思路提示独立实现；卡住再对照教师课堂演示 |
| **B/A 组长·师傅** | 不看提示直接做；完成后做扩展：把 `batch` 改 10 看批量阈值效果、给 A 层加"成功率/延迟/丢弃"实时小面板，写进工单"参数-收益"留痕 |

**报错自查三板斧**：①读最后一行错误信息 → ②检查变量名/缩进/括号 → ③还不行问同层同学或教师
**自检**：对照上方“验收标准”四行逐条自查并在工单上签字，未做到的档位如实写「未做」——不得只勾 及格(D) 与 优秀(B+)：`及格(D)`＝`三源数据贯通；采集率>95%；无数据丢失；系统文档` ｜ `良好(C)`＝`断线重连实测通过；统计输出准确` ｜ `优秀(B)`＝`采集率≥99.5%；延迟<100ms；线程安全(无共享可变状态)` ｜ `超优(A)`＝`优秀+故障自动转移/主从切换或性能监控小面板`


### Starter 3：`class MQTTSubscriber(threading.Thread)`
（教师版：下方为完成态可运行代码；学生版此处为待填空位）


In [11]:
class MQTTSubscriber(threading.Thread):
    """任务2(学生版第2空): 订阅 device/+/status, 消息JSON解析后放入Queue"""

    def __init__(self, q):
        super().__init__(daemon=True, name="MQTT")
        self.q = q
        self.stat = {"recv": 0, "drop": 0}      # 教师版补充: 对账用计数(工单工程化要求②)
        self._c = None

    def run(self):
        # paho 2.x 建议显式 mqtt.Client(mqtt.CallbackAPIVersion.VERSION1) 以保持三参回调签名
        c = mqtt.Client()
        self._c = c

        def on_message(_c, _u, msg):
            # ===== 任务2 教师版完成态: json.loads → dict加source="mqtt" → q.put =====
            try:
                payload = json.loads(msg.payload.decode("utf-8"))   # bytes → str → dict
                data = dict(payload.get("metrics", {}))             # 摊平一层: 与PLC同键名口径
                data["device_id"] = payload.get("device_id", "unknown")
                data["source"] = "mqtt"                              # 来源标记(去重键的另一半)
                data["topic"] = msg.topic
                data["ts"] = _to_epoch(payload.get("ts"))            # 必须数值秒, 否则下游去重报错
                self.stat["recv"] += 1
                try:
                    self.q.put(data, timeout=1)                      # 禁裸put: 带超时防永久阻塞
                except queue.Full:
                    self.stat["drop"] += 1                           # 背压丢弃计入对账
            except Exception as e:                                   # 脏报文只丢这一条, 线程不能死
                log.warning(f"MQTT报文解析失败(已丢弃): {type(e).__name__}: {e}")

        c.on_message = on_message
        c.connect("127.0.0.1", 1883)
        c.subscribe("device/+/status", qos=0)      # "+" 通配一层 = 三台设备都收
        c.loop_forever()                           # 阻塞收报文; daemon线程随主程序退出
        log.info(f"MQTT线程收尾: 收到{self.stat['recv']} 丢弃{self.stat['drop']}")

    def stop(self):
        """教师版补充: 优雅收尾 —— 真实 paho 走 disconnect, 离线模拟源置事件自停
        (notebook 反复运行不会叠一堆长驻线程, 也方便 p41 自检里控时)"""
        c = self._c
        if c is None:
            return
        try:
            (c.disconnect if USE_MQTT else c.stop)()
        except Exception as e:
            log.warning(f"MQTT关闭异常(忽略): {e}")


### Starter 4：`class DataProcessor(threading.Thread)`
（教师版：下方为完成态可运行代码；学生版此处为待填空位）


In [13]:
class DataProcessor(threading.Thread):
    """任务3(学生版第3空): 消费Queue → 去重(同source+ts跳过) → 振动>4.5告警 → 批量入库SQLite"""

    def __init__(self, q, batch=50, db_path=DB_PATH):
        super().__init__(daemon=True, name="Proc")
        self.q, self.batch, self.buf = q, batch, []
        self.db_path = db_path
        self.stop = threading.Event()            # 教师版补充: 可优雅退出(要落库的线程不能只靠daemon)
        self.stat = {"recv": 0, "dup": 0, "alarm": 0, "rows": 0}
        self.t_flush = time.time()
        self.lat = []                            # A层简易APM: 端到端延迟样本

    def take(self, d, seen, db):
        """单条记录处理: 幂等去重 → 超限告警 → 进缓冲 → 满batch/超5秒就批量入库
        返回 True=入缓冲, False=重复丢弃 (抽成方法便于单测与收尾排空复用)"""
        self.stat["recv"] += 1
        base_key = (d.get("source"), round(d.get("ts", 0), 1))    # 骨架已给的去重键: 同来源+同时间戳
        # 教师版补充: mqtt 这一路三台设备会落进同一个0.1秒桶 → 键里补 device_id 才不误伤(幂等键要唯一标识一条数据)
        key = base_key + (d.get("device_id", ""),)
        if key in seen:                                       # 命中 = 双发/重投 → 丢弃并计数
            self.stat["dup"] += 1
            return False
        seen.add(key)
        ts = float(d.get("ts", 0.0))
        temp = float(d.get("temp_c", d.get("temp", 0.0)) or 0.0)
        vib = float(d.get("vib_mm_s", d.get("vib", 0.0)) or 0.0)
        self.lat.append(time.time() - ts)                     # 采集→入库 延迟(B层要求<100ms)
        if vib > VIB_LIMIT:                                   # 告警: 振动>4.5mm/s(工单阈值)
            self.stat["alarm"] += 1
            log.warning(f"⚠告警 {d.get('source')} 振动 {vib:.2f} > {VIB_LIMIT} mm/s "
                        f"(温度 {temp:.1f}℃) → 记录已入库待人工确认")
        self.buf.append((d.get("source"), ts, temp, vib))
        if len(self.buf) >= self.batch or time.time() - self.t_flush >= FLUSH_SECONDS:
            self.flush(db)                                    # 每50条 或 每5秒 写一次SQLite
        return True

    def run(self):
        db = sqlite3.connect(self.db_path, check_same_thread=False)
        db.execute("CREATE TABLE IF NOT EXISTS records("
                   "source TEXT, ts REAL, temp REAL, vib REAL)")
        seen = set()
        while not self.stop.is_set():                         # 教师版: while True → 可优雅退出
            try:
                d = self.q.get(timeout=GET_TIMEOUT)
            except queue.Empty:                               # 心跳超时(骨架给的防死等写法)
                if self.buf and time.time() - self.t_flush >= FLUSH_SECONDS:
                    self.flush(db)                            # 断流兜底: 每5秒落盘一次
                continue
            self.take(d, seen, db)
        while True:                                           # 收尾: 排空在途数据 = 验收D"无数据丢失"
            try:
                self.take(self.q.get_nowait(), seen, db)
            except queue.Empty:
                break
        self.flush(db)                                        # 退出前必须落盘, 否则buf里的丢数
        db.close()
        log.info(f"Proc线程收尾: 收到{self.stat['recv']} 去重{self.stat['dup']} "
                 f"告警{self.stat['alarm']} 落库{self.stat['rows']}")

    def flush(self, db):
        if self.buf:
            n = len(self.buf)
            db.executemany("INSERT INTO records VALUES(?,?,?,?)", self.buf)   # 批量写=一次事务
            db.commit()
            self.stat["rows"] += n
            self.t_flush = time.time()
            log.info(f"批量入库 {n} 条 → {self.db_path} "
                     f"(阈值 batch={self.batch} / {FLUSH_SECONDS:.0f}s)")
            self.buf.clear()


### Starter 5：`main()/reconcile()/report_status()`
（教师版：下方为完成态可运行代码；学生版此处为待填空位）


In [15]:
def reconcile(mc, mq, dp, q):
    """工单工程化要求②: 对账恒等式 —— 发送 = 接收 + 在途 + 丢弃, 一条都不许下落不明"""
    sent = mc.stat["ok"] + mq.stat["recv"]
    dropped = mc.stat["drop"] + mq.stat["drop"]
    inflight = q.qsize()
    balance = sent - (dp.stat["recv"] + inflight + dropped)
    if balance:
        log.error(f"对账不平: 差额 {balance} 条 → 检查是否漏了计数分支")
    return {"sent": sent, "inflight": inflight, "drop": dropped, "balance": balance,
            **dp.stat}


def main(seconds=None):
    seconds = RUN_SECONDS if seconds is None else float(seconds)
    q = queue.Queue(maxsize=10000)                     # 带背压的传送带(骨架已给)
    mc, mq, dp = ModbusCollector(q), MQTTSubscriber(q), DataProcessor(q)
    t0 = time.time()                                   # 起点要在开线程之前取: 本次新增行都 ts>=t0 才对得上账
    for t in (mc, mq, dp):
        t.start()                                      # 三线程并发: 采集/订阅/入库互不挡道
    print(f"采集 {seconds:.0f} 秒后输出统计(骨架原意是30秒——把这里改成 time.sleep(30) 即课堂行为, "
          f"Ctrl+C 提前结束)...")
    try:
        time.sleep(seconds)
    except KeyboardInterrupt:
        pass
    mc.stop.set()                                      # 停采集(骨架已给)
    mq.stop()                                          # 教师版补充: 停订阅源(离线模拟源自停)
    mq.join(timeout=PUB_INTERVAL * 2)                  # 真停之后再对账, 否则统计窗口里还有新报文进来
    dp.stop.set()
    dp.join(timeout=GET_TIMEOUT + 3)                   # 等它排空+落盘(不join就可能丢最后一批buf)
    s = mc.stat
    total = s["ok"] + s["fail"]
    rate = s["ok"] / max(total, 1)                     # 采集率 = 成功次数 / 应采次数
    rec = reconcile(mc, mq, dp, q)
    lat_avg = sum(dp.lat) / len(dp.lat) if dp.lat else 0.0
    lat_max = max(dp.lat) if dp.lat else 0.0
    print(f"采集率: {s['ok']}/{total} = {rate:.1%} (目标≥95%)  "
          f"退避重试 {s['retry']} 次"
          + (" ← 虚拟PLC每" + str(OUTAGE_EVERY) + "次连接必拒1次, 重连路径已实测"
             if not USE_MODBUS else
             " ← 真PLC: 掉线由模拟器守护注入(45~90s一次), 5秒窗口未命中即为0"))
    print(f"数据源: Modbus 入队 {s['ok']} 条 / MQTT 收到 {mq.stat['recv']} 条(命中去重键丢弃 {rec['dup']} 条)")
    print(f"入库: {rec['rows']} 条 → {DB_PATH} | 振动告警 {rec['alarm']} 条 | 队列积压 {rec['inflight']}")
    print(f"对账: 发送 {rec['sent']} = 接收 {rec['recv']} + 在途 {rec['inflight']} "
          f"+ 丢弃 {rec['drop']} (差额 {rec['balance']})")
    print(f"APM: 平均延迟 {lat_avg * 1000:.1f}ms / 峰值 {lat_max * 1000:.1f}ms (B层要求<100ms)")
    return {"rate": rate, "t0": t0, "stat": s, "lat_avg": lat_avg, "lat_max": lat_max, **rec}


def report_status(res):
    """A层拓展: 一行式采集健康面板(可挂到 Streamlit/Grafana 轮询)"""
    level = "🟢正常" if res["rate"] >= 0.99 and res["balance"] == 0 else \
            "🟡关注" if res["rate"] >= 0.95 else "🔴告警"
    print(f"{level} 采集率{res['rate']:.1%} | 入库{res['rows']} | 去重{res['dup']} | "
          f"告警{res['alarm']} | 对账差额{res['balance']}")


### Starter 6：`if __name__ == "__main__"`
（教师版：下方为完成态可运行代码；学生版此处为待填空位）


In [17]:
if __name__ == "__main__":
    # 教师版完成态: 存一份统计结果, 供下一 cell 逐条做验收断言(脚本运行时等价于只看打印)
    RESULT = main()


[离线MQTT] 已连接进程内代理 127.0.0.1:1883(真实环境先起 mini_mqtt_broker.py)
采集 5 秒后输出统计(骨架原意是30秒——把这里改成 time.sleep(30) 即课堂行为, Ctrl+C 提前结束)...
03:46:29.587 Proc    WARNING ⚠告警 plc@5020 振动 4.90 > 4.5 mm/s (温度 67.6℃) → 记录已入库待人工确认
03:46:29.587 Proc    WARNING ⚠告警 plc@5022 振动 5.10 > 4.5 mm/s (温度 57.0℃) → 记录已入库待人工确认
03:46:30.098 Proc    WARNING ⚠告警 plc@5022 振动 5.30 > 4.5 mm/s (温度 56.7℃) → 记录已入库待人工确认
03:46:30.393 Proc    WARNING ⚠告警 mqtt 振动 4.93 > 4.5 mm/s (温度 72.9℃) → 记录已入库待人工确认
03:46:30.606 Modbus  WARNING PLC@5021 第1次连接被拒(掉线)
03:46:30.607 Modbus  WARNING PLC@5021 退避 1.00s 后重试
03:46:31.198 Proc    WARNING ⚠告警 mqtt 振动 5.24 > 4.5 mm/s (温度 71.1℃) → 记录已入库待人工确认
03:46:31.618 Modbus  INFO PLC@5021 第2次尝试重连成功 ✓
03:46:31.618 Proc    WARNING ⚠告警 plc@5021 振动 5.00 > 4.5 mm/s (温度 67.9℃) → 记录已入库待人工确认
03:46:32.003 Proc    WARNING ⚠告警 mqtt 振动 4.71 > 4.5 mm/s (温度 55.4℃) → 记录已入库待人工确认
03:46:32.004 Proc    WARNING ⚠告警 mqtt 振动 5.04 > 4.5 mm/s (温度 72.2℃) → 记录已入库待人工确认
03:46:32.127 Proc    WARNING ⚠告警 plc@5021 振动 5.30 > 4.5 mm/s 

### 教师版演示：把运行统计变成"验收断言"
工单验收三档（D 三源贯通+采集率>95% / C 统计准确 / B 断线重连实测）在这里**用断言机器化**：
采集率、对账恒等式、去重条数、告警条数、以及"库内新增行数 == 统计入库数"。
`_executed.ipynb` 里保留的就是这些断言全绿的真实输出，可直接对着学生演示。


In [19]:
# 教师版完成态: 三线程运行结果自检(对应工单 D/C/B/A 四档验收)
assert RESULT["rate"] >= 0.95, f"采集率未达标: {RESULT['rate']:.1%}"
assert RESULT["balance"] == 0, f"对账不平: {RESULT['balance']} 条下落不明"
assert RESULT["sent"] == RESULT["recv"] + RESULT["inflight"] + RESULT["drop"], "恒等式不成立"
assert RESULT["recv"] == RESULT["dup"] + RESULT["rows"], "接收 = 去重丢弃 + 入库(不许丢数)"
assert RESULT["rows"] > 0, "应有数据入库"
# 下面三条依赖"故障注入"(虚拟PLC每3次拒1次 / 模拟broker双发 / 振动越限正弦) —— 离线口径必现,
# 真协议口径由 工具/plc_simulator.py 的掉线守护与现场工况决定, 5秒窗口未必命中, 故只报数不判死。
if not USE_MQTT:
    assert RESULT["dup"] >= 1, "双发报文应被幂等去重"
if not USE_MODBUS:
    assert RESULT["stat"]["retry"] >= 1, "2号PLC掉线应触发过退避重试(B层实测点)"
    assert RESULT["alarm"] >= 1, "振动>4.5mm/s 应触发告警"
if USE_MODBUS or USE_MQTT:
    print(f"真协议口径故障命中情况: 去重{RESULT['dup']} 告警{RESULT['alarm']} "
          f"退避重试{RESULT['stat']['retry']} (为0不代表重连逻辑没写, 见 p41 第3步拒连单测)")

if USE_MODBUS:                                 # 真协议口径必做：核对"寄存器→工程值"是否符合点表量程
    _SPECS = {"temp_c": (20, 90, "℃"), "press_mpa": (0.1, 0.9, "MPa"),
              "vib_mm_s": (1.0, 8.0, "mm/s")}
    _s = ModbusCollector(queue.Queue()).read_one("127.0.0.1", PLCS[0][1])
    for _k, (_lo, _hi, _u) in _SPECS.items():
        _ok = _lo <= _s[_k] <= _hi
        print(f"  点表核对 {_k}={_s[_k]}{_u} 应在 {_lo}~{_hi}{_u} → " + ("✓" if _ok else "✗ 越界"))
    print("  越界≠采集代码写错：先核对「SCALE 定点口径」(例: mini_modbus_server 的 40002 压力把"
          "已经 ×10 的值又 ×10, ÷10 后得 3~7MPa) —— 改模拟器/点表, 别改验收断言。")
assert RESULT["lat_max"] < 1.0, f"端到端延迟异常: {RESULT['lat_max']:.2f}s"

con = sqlite3.connect(DB_PATH)
n_db = con.execute("SELECT COUNT(*) FROM records WHERE ts >= ?", (RESULT["t0"],)).fetchone()[0]
sources = [r[0] for r in con.execute("SELECT DISTINCT source FROM records WHERE ts >= ?",
                                      (RESULT["t0"],)).fetchall()]
con.close()
assert n_db == RESULT["rows"], f"库内新增 {n_db} ≠ 统计入库 {RESULT['rows']}"
assert any(s.startswith("plc@") for s in sources), "PLC 源未落库"
if not USE_MQTT:                                     # 离线口径自带发布源; 真 broker 需另有设备在发
    assert "mqtt" in sources, "三源未贯通(MQTT 源缺失)"
report_status(RESULT)
print(f"✓ 断言全绿: 采集率{RESULT['rate']:.1%} | 入库{n_db}条(buffer.db) | 来源{sorted(sources)}")


🟢正常 采集率100.0% | 入库39 | 去重9 | 告警16 | 对账差额0
✓ 断言全绿: 采集率100.0% | 入库39条(buffer.db) | 来源['mqtt', 'plc@5020', 'plc@5021', 'plc@5022']


## 四、提交要求与避坑
- 提交: `学号_姓名_P4.1.py` 或本notebook(含输出)
- ⚠ 线程命名：threading.Thread(name='Modbus')日志可辨
- ⚠ Queue.put(timeout=1)防永久阻塞
- ⚠ daemon=True保证Ctrl+C能退出；**但要 flush 的入库线程得给 stop 事件优雅收尾**，否则最后一批数据留在内存里丢数
- ⚠ pymodbus 3.x：`Client(host=…, port=…)` 与 `read_holding_registers(address=…, count=…)` 用关键字参数
- ⚠ 去重键里的 `ts` 必须是数值秒：报文里给的是 `"HH:MM:SS"` 字符串 → 先 `_to_epoch()` 归一
- ⚠ 寄存器是 ×10 定点：÷10 还原工程值，`status` 状态字不除（点表口径要写进工单）


---

## 教师参考答案（与 `教师参考/solutions.py` · `p41()` 同源，本 notebook 内已可运行）

> **发放纪律：本附录勿随学生版下发。** 课堂按层放提示（D层演示照做/C层只给思路行/A层直接做）。

学生版 3 处待填空位 → 下方参考实现的对应关系：

| 学生版待填空位 | 所在函数 | 教师完成态所在 cell | 关键实现要点 |
|---|---|---|---|
| 任务1 轮询读寄存器 + 退避重连 | `ModbusCollector.read_one()` | Starter 2（cell 7） | `Client(host=,port=)` → `connect()` → `read_holding_registers(address=0,count=5)`；`isError()` 判异常；÷10 还原（status 除外）；`2**i` = 1s→2s→4s；`finally: c.close()`；失败返回 None |
| 任务2 MQTT 报文解析入队 | `MQTTSubscriber.on_message()` | Starter 3（cell 10） | `json.loads(msg.payload.decode())` → `metrics` 摊平 + `source="mqtt"` + `_to_epoch(ts)` → `q.put(timeout=1)`，`except queue.Full` 计丢弃；整段 try 防脏报文弄死线程 |
| 任务3 去重/告警/批量入库 | `DataProcessor.run()` | Starter 4（cell 12） | `(source, round(ts,1))`(+`device_id`) 命中即跳过；`vib>4.5` → `log.warning`+计数；`buf` 满 50 条或超 5 秒 `flush()`；退出前排空队列 + 末次 flush + `db.close()` |
| 运行时长与统计输出 | `main()` | Starter 5（cell 14） | `RUN_SECONDS`（notebook 5s / 课堂 30s）；采集率、对账恒等式、延迟与丢弃一次打全 |
| 优雅收尾（骨架外的工程化补强） | `MQTTSubscriber.stop()` / `DataProcessor.stop` | cell 10 / cell 12 | 要落库的线程给 Event 停 + `join()`，不靠 daemon 硬杀 |
| 离线降级（本机无 pymodbus/paho） | 顶部导入块 | Starter 1（cell 5） | 进程内 `SimModbusClient`（×10 定点寄存器 + 2号PLC每4次拒1次）与 `SimMqttClient`（`device/+/status` JSON + 每5轮双发） |
| 端到端参考答案 | `p41()` | cell 21 | 单点÷10校验 + 幂等去重单测 + 拒连退避单测 + 三线程短跑 + 对账/入库断言 |

> 下方 `p41()` 为**可直接运行**的参考答案（纯离线、自足、含断言），不再依赖仓库外的模拟器脚本。
> 注：表内 cell 编号为本教师版 notebook 的当前序号，讲解时按 Starter 小标题定位更稳。


In [22]:
def _p41_body(seconds=3.0):
    """P4.1 参考实现四步: ÷10还原 / 幂等去重 / 退避重连 / 端到端对账入库"""
    global Client, BACKOFF_UNIT          # 第3步要临时替换协议库/缩放退避, 声明global才能改模块级名字

    # 1) 单点校验: read_one 的 ×10 定点还原与来源标记
    probe = queue.Queue()
    d = ModbusCollector(probe).read_one("127.0.0.1", 5022)
    assert d and d["source"] == "plc@5022", "read_one 必须带来源端口"
    assert isinstance(d["status"], int) and d["status"] in (0, 1), "状态字不除10"
    assert 30 < d["temp_c"] < 90 and 0.1 <= d["press_mpa"] <= 1.0, "÷10 后应在物理量程内"
    assert 1.0 <= d["vib_mm_s"] <= 8.0, "振动还原口径 3.0~5.4mm/s"
    print(f"  1) 寄存器÷10还原 ✓ {d['temp_c']:.1f}℃ {d['press_mpa']:.2f}MPa "
          f"{d['vib_mm_s']:.2f}mm/s rpm_raw{d['rpm']} status{d['status']}")

    # 2) 幂等去重 + 告警单测(内存库, 不污染 buffer.db): 同一条记录喂 3 次
    seen, db = set(), sqlite3.connect(":memory:")
    db.execute("CREATE TABLE IF NOT EXISTS records(source TEXT, ts REAL, temp REAL, vib REAL)")
    dp_u = DataProcessor(queue.Queue(), batch=5, db_path=":memory:")
    row = {"source": "plc@5022", "ts": 1700000000.0, "temp_c": 61.0, "vib_mm_s": 5.20}
    flags = [dp_u.take(dict(row), seen, db) for _ in range(3)]
    dp_u.flush(db)
    assert flags == [True, False, False] and dp_u.stat["dup"] == 2, "重复投递应被跳过并计数"
    assert dp_u.stat["alarm"] == 1 and dp_u.stat["rows"] == 1
    assert db.execute("SELECT COUNT(*) FROM records").fetchone()[0] == 1
    db.close()
    print("  2) 幂等去重(3投1存) + 振动告警计数 ✓")

    # 3) 断线重连单测: 全部拒连 → 1s→2s→4s 退避(演示时缩放0.05倍)后必须返回 None 而非抛异常
    saved_client, saved_unit = Client, BACKOFF_UNIT

    class _Dead:
        def __init__(self, host="127.0.0.1", port=5020, **kw):
            self.port = port

        def connect(self):
            return False

        def close(self):
            pass

    Client, BACKOFF_UNIT = _Dead, 0.05
    try:
        mc_u = ModbusCollector(queue.Queue())
        assert mc_u.read_one("127.0.0.1", 5999) is None, "彻底拒连时返回None(交给上层退避)"
        assert mc_u.stat["retry"] == RETRY_TIMES - 1, "退避应为 1s→2s(共2次等待, 封顶4s)"
    finally:
        Client, BACKOFF_UNIT = saved_client, saved_unit
    print(f"  3) 退避重连 ✓ ({RETRY_TIMES}次尝试, 间隔 2**i 封顶{BACKOFF_CAP}s) 单元耗时<0.5s")

    # 4) 端到端三线程短跑(离线): 采集率/对账/入库行数三者咬合
    res = main(seconds=seconds)
    assert res["rate"] >= 0.95, f"虚拟产线采集率应≥95%, 实测 {res['rate']:.1%}"
    assert res["balance"] == 0 and res["sent"] == res["recv"] + res["inflight"] + res["drop"]
    assert res["recv"] == res["dup"] + res["rows"], "无数据丢失(验收D硬指标)"
    assert res["rows"] > 0 and res["dup"] >= 1 and res["alarm"] >= 1
    con = sqlite3.connect(DB_PATH)
    n_db = con.execute("SELECT COUNT(*) FROM records WHERE ts >= ?", (res["t0"],)).fetchone()[0]
    srcs = sorted({r[0] for r in con.execute(
        "SELECT DISTINCT source FROM records WHERE ts >= ?", (res["t0"],))})
    con.close()
    assert n_db == res["rows"], f"库内新增 {n_db} ≠ 统计 {res['rows']}"
    assert any(s.startswith("plc@") for s in srcs) and "mqtt" in srcs, "三源未贯通"
    print(f"  4) 端到端 ✓ 采集率{res['rate']:.1%} 发送{res['sent']}=接收{res['recv']}"
          f"+在途{res['inflight']}+丢弃{res['drop']} | 入库{n_db}条 | 去重{res['dup']} "
          f"| 告警{res['alarm']} | 来源{srcs}")
    print(f"本次口径: 进程内虚拟产线(SimModbusClient/SimMqttClient) | 本机真库: pymodbus={'已装' if HAVE_MODBUS else '未装'}, "
          f"paho={'已装' if HAVE_MQTT else '未装'} | notebook 探活结果: USE_MODBUS={USE_MODBUS}, USE_MQTT={USE_MQTT}")
    print("P4.1 参考答案断言全部通过 ✓  (D=三源贯通+采集率>95% / C=统计准确 / B=断线重连 / A=APM面板)")
    return res


def p41(seconds=3.0):
    """教师参考答案(P4.1): 三线程端到端最小闭环 + 四组断言(÷10还原/幂等去重/退避重连/对账入库)
    —— 固定走进程内虚拟产线: 不占端口、不依赖仓库外脚本, 因此在任何机器上反复跑都能得到同一验收结论。
       真协议口径请看 cell 5 探活 + cell 16/18: 起好 plc_simulator.py 与 mini_mqtt_broker.py 后
       Client/mqtt 自动换成 pymodbus/paho, read_one 与 on_message 的实现一行都不用改。"""
    global Client, mqtt
    saved_client, saved_mqtt = Client, mqtt
    Client, mqtt = SimModbusClient, _MqttModule()
    print("—— p41 参考实现开始 —— 口径: 进程内虚拟产线(端口无关, 可反复跑)")
    try:
        return _p41_body(seconds)
    finally:
        Client, mqtt = saved_client, saved_mqtt


p41()


—— p41 参考实现开始 —— 口径: 进程内虚拟产线(端口无关, 可反复跑)
  1) 寄存器÷10还原 ✓ 52.7℃ 0.50MPa 5.20mm/s rpm_raw14.8 status1
03:46:35.447 MainThread WARNING ⚠告警 plc@5022 振动 5.20 > 4.5 mm/s (温度 61.0℃) → 记录已入库待人工确认
03:46:35.448 MainThread INFO 批量入库 1 条 → :memory: (阈值 batch=5 / 5s)
  2) 幂等去重(3投1存) + 振动告警计数 ✓
03:46:35.448 MainThread WARNING PLC@5999 第1次连接被拒(掉线)
03:46:35.448 MainThread WARNING PLC@5999 退避 0.05s 后重试
03:46:35.505 MainThread WARNING PLC@5999 第2次连接被拒(掉线)
03:46:35.506 MainThread WARNING PLC@5999 退避 0.10s 后重试
03:46:35.615 MainThread WARNING PLC@5999 第3次连接被拒(掉线)
  3) 退避重连 ✓ (3次尝试, 间隔 2**i 封顶4s) 单元耗时<0.5s
03:46:35.616 Modbus  WARNING PLC@5021 第1次连接被拒(掉线)
03:46:35.616 Modbus  WARNING PLC@5021 退避 1.00s 后重试
[离线MQTT] 已连接进程内代理 127.0.0.1:1883(真实环境先起 mini_mqtt_broker.py)
采集 3 秒后输出统计(骨架原意是30秒——把这里改成 time.sleep(30) 即课堂行为, Ctrl+C 提前结束)...
03:46:35.618 Proc    WARNING ⚠告警 mqtt 振动 5.14 > 4.5 mm/s (温度 40.9℃) → 记录已入库待人工确认
03:46:35.618 Proc    WARNING ⚠告警 mqtt 振动 5.21 > 4.5 mm/s (温度 58.7℃) → 记录已入库待人工确认
03:46:36.427 Proc 

## 运行与验证

**本教师版已在课程环境 `/opt/anaconda3/envs/pp03/bin/python`（Python 3.12，pandas 3.0.6 / pymodbus 3.15.0 / paho-mqtt 已装）逐 cell 跑通。**
该环境里 pymodbus、paho-mqtt **都装了**，但课堂机上 PLC 与 broker 未必在跑，所以 cell 5 用 TCP 探活决定口径：
本次 `_executed.ipynb` 的探活结果是 5020~5022 与 1883 全部无响应 → 走**进程内虚拟产线**，输出里 `PLC 数据源` / `MQTT 数据源` 两行已如实标注。

| cell | 内容 | 执行结果 |
|---|---|---|
| 2 | `os.chdir("../")` 切到课程包根 | `buffer.db` 落在课程包根 |
| 5 | 导入 + 参数 + **探活选源**(`USE_MODBUS/USE_MQTT`) | 打印库状态 + 两行数据源口径 + 切真协议的方法 |
| 7 | `ModbusCollector.read_one/run` 完成态 | 三台PLC轮询、÷10 还原、1s→2s→4s 退避重连 |
| 10 | `MQTTSubscriber.on_message` 完成态 | `device/+/status` JSON → 摊平 + `source="mqtt"` → 入队 |
| 12 | `DataProcessor.take/run` 完成态 | `(source, ts)` 幂等去重、`vib>4.5` 告警、每50条/每5秒批量入库、收尾排空 |
| 14 | `main()/reconcile()/report_status()` | 采集率、对账恒等式、延迟/丢弃 APM 行 |
| 16 | 脚本入口守卫 | 真实运行 `main()`，结果存入 `RESULT`（5 秒三线程并发） |
| 18 | 验收断言 | 硬不变量(采集率/对账/无丢失/来源)全断言 + 故障注入项按口径断言 |
| 21 | `p41()` 参考答案 | ÷10 / 去重 / 退避 / 端到端 四组断言全绿；钉离线口径，跑完自动恢复 |

**重跑方式**
- Jupyter：`Kernel → Restart` 后 `Run All`（约 13 秒：cell 16 采 5s + cell 21 采 3s，另含退避重连与收尾等待）。
- 命令行等价：`/opt/anaconda3/envs/pp03/bin/python /tmp/qoder_nb/capture.py W06_P4.1_多源数据采集系统_教师版.ipynb W06_P4.1_多源数据采集系统_教师版_executed.ipynb Jupyter实训`
- **课堂真机（推荐，D/C 层用真 PLC 读数）**：
  ```bash
  python 工具/plc_simulator.py            # 三台PLC: 5020/5021/5022，2号每45~90秒掉线5秒(断线重连实测)
  python ../02-程序设计基础/虚拟产线/mini_mqtt_broker.py    # broker:1883
  ```
  再起一个设备发布者（`02-程序设计基础/虚拟产线/` 内的上报脚本，或 cell 5 之后临时把 `SimMqttClient` 当发布者用），
  然后 `Run All`：cell 5 探活成功 → `Client` 自动换成 `pymodbus.ModbusTcpClient`、`mqtt` 换成真 `paho`，
  **代码一行不改**；此时 `USE_MODBUS=True`，退避重试是否为 0 取决于掉线守护是否命中，cell 18 会照实报数不判死。
- 查看结果：`sqlite3 buffer.db "SELECT source, COUNT(*) FROM records GROUP BY source;"`
  （每次运行都按 `ts >= 本次起始时间` 统计，历史数据不干扰断言）。

> 多解法对比见同目录《多解法手册.md》与 `教师参考/solutions_multi*.py`。
